# Early Sepsis Prediction: time-aware model with calibrated probabilities

Deep Data Hackathon 2026, problem statement 1. For every ICU hour the model outputs the probability that the patient is developing sepsis.

**Approach**
1. **Cleaning** (causal, per patient): drop the empty `EtCO2` column, blank out physiologically impossible values, forward-fill only (never backfill), and add measurement-count and time-since-last-measurement features, because missingness is informative.
2. **Features** (strictly causal, no future hours): rolling 6 h and 12 h statistics, 6 h and 12 h trends, running extremes since ICU admission, and qSOFA / SIRS / partial SOFA style scores. A unit test checks that changing future hours never changes an earlier feature.
3. **Validation**: 5-fold `GroupKFold` on `Patient_ID`, so no patient appears on both sides. Metrics (AUPRC, AUROC) come from out-of-fold predictions.
4. **Calibration and threshold**: Platt scaling is fitted on out-of-fold predictions and kept only if it improves the Brier score on held-out patients. The alert threshold maximises F1 on the calibrated out-of-fold probabilities.
5. **Submission**: the model is refit on all training patients and scores every test row. The file is validated against `sample_submission.csv` (same columns, rows and order, probabilities in [0, 1], no missing rows).

In [ ]:
import json, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

FAST = os.environ.get("SEPSIS_FAST") == "1"   # smoke test only; never set on Kaggle
ON_KAGGLE = Path("/kaggle/input").exists()
OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()


def find(name):
    """Locate a competition file under /kaggle/input, or data/raw when running locally."""
    if ON_KAGGLE:
        hits = sorted(Path("/kaggle/input").rglob(name))
        if hits:
            return hits[0]
    for base in [Path.cwd(), *Path.cwd().parents]:
        for rel in ("data/raw", "."):
            if (base / rel / name).exists():
                return base / rel / name
    raise FileNotFoundError(f"{name} not found. Add the dataset to the notebook (Add Input) and re-run.")


PATHS = {n: find(n) for n in ("train.csv", "test.csv", "sample_submission.csv")}
print({k: str(v) for k, v in PATHS.items()})
os.makedirs(OUT_DIR / "sepsis_src" / "sepsis", exist_ok=True)

## Source modules
The cells below write the pipeline's modules to disk (identical to `src/sepsis/` in the project repository) and import them.

In [ ]:
%%writefile sepsis_src/sepsis/__init__.py
# sepsis pipeline package


In [ ]:
%%writefile sepsis_src/sepsis/config.py
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
SUBMISSIONS = ROOT / "submissions"

ID, TIME, TARGET = "Patient_ID", "Hour", "SepsisLabel"
VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp"]
LABS = ["BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN", "Alkalinephos", "Calcium",
        "Chloride", "Creatinine", "Bilirubin_direct", "Glucose", "Lactate", "Magnesium", "Phosphate",
        "Potassium", "Bilirubin_total", "TroponinI", "Hct", "Hgb", "PTT", "WBC", "Fibrinogen", "Platelets"]
DYNAMIC = VITALS + LABS
DROP_COLS = ["EtCO2"]  # 100% missing
# plausible physiological ranges; outside -> NaN
RANGES = {"HR": (20, 300), "O2Sat": (20, 100), "Temp": (25, 45), "SBP": (30, 300), "MAP": (20, 300),
          "DBP": (10, 300), "Resp": (2, 80), "FiO2": (0.1, 1.0), "pH": (6.5, 8.0), "Hct": (5, 70),
          "Hgb": (2, 25), "Glucose": (10, 1500)}
SEED = 42
N_FOLDS = 5


In [ ]:
%%writefile sepsis_src/sepsis/cleaning.py
"""Causal, per-patient cleaning: nothing here looks at future hours."""
import numpy as np
import pandas as pd

from .config import DROP_COLS, DYNAMIC, ID, RANGES, TIME


def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values([ID, TIME]).reset_index(drop=True)
    df = df.drop(columns=DROP_COLS, errors="ignore")
    for c, (lo, hi) in RANGES.items():
        df.loc[(df[c] < lo) | (df[c] > hi), c] = np.nan
    # Unit1/Unit2 both NaN for one hospital system: keep that as its own signal
    df["Unit_missing"] = df.Unit1.isna().astype(int)
    df[["Unit1", "Unit2"]] = df[["Unit1", "Unit2"]].fillna(0)
    obs = df[DYNAMIC].notna()
    new = {}
    for c in DYNAMIC:
        new[c + "_obs"] = obs[c].astype(np.int8)
        last = df[TIME].where(obs[c]).groupby(df[ID]).ffill()
        new[c + "_tsl"] = (df[TIME] - last).fillna(-1)  # hours since last measured, -1 = never
    new = pd.DataFrame(new)
    cnt = obs.groupby(df[ID]).cumsum().add_suffix("_cnt")
    df = pd.concat([df, new, cnt], axis=1)
    df[DYNAMIC] = df.groupby(ID)[DYNAMIC].ffill()  # forward-fill only, never backfill
    return df


In [ ]:
%%writefile sepsis_src/sepsis/features.py
"""Temporal features. Every feature at hour t uses only hours <= t of the same patient (no future data, no other patients).

Input is the cleaned frame from cleaning.clean(): forward-filled vitals/labs plus *_obs / *_tsl / *_cnt columns.
"""
import numpy as np
import pandas as pd

from .config import DYNAMIC, ID, TARGET, TIME, VITALS

DEMOGRAPHICS = ["Age", "Gender", "Unit1", "Unit2", "Unit_missing", "HospAdmTime", "ICULOS"]
WINDOW = 6    # hours for the short rolling statistics
LONG_WINDOW = 12
LAG = 6       # hours for trend features
LONG_LAG = 12
# running extremes since ICU admission: (column, cummax | cummin)
RUNNING = [("HR", "cummax"), ("Temp", "cummax"), ("Temp", "cummin"), ("Resp", "cummax"), ("MAP", "cummin"), ("SBP", "cummin"),
           ("Lactate", "cummax"), ("Creatinine", "cummax"), ("WBC", "cummax"), ("WBC", "cummin"), ("Platelets", "cummin"),
           ("Bilirubin_total", "cummax")]


def clinical_scores(df: pd.DataFrame) -> pd.DataFrame:
    """qSOFA / SIRS / partial SOFA style scores from the latest known values. Unknown values count as 'not met'."""
    d = df
    resp = d["Resp"]
    qsofa = (resp >= 22).astype(int) + (d["SBP"] <= 100).astype(int)  # GCS is not in this dataset
    sirs = (((d["Temp"] > 38) | (d["Temp"] < 36)).astype(int) + (d["HR"] > 90).astype(int)
            + ((resp > 20) | (d["PaCO2"] < 32)).astype(int) + ((d["WBC"] > 12) | (d["WBC"] < 4)).astype(int))
    cr, plt, bil = d["Creatinine"], d["Platelets"], d["Bilirubin_total"]
    sofa = ((d["MAP"] < 70).astype(int)
            + np.select([cr >= 5, cr >= 3.5, cr >= 2, cr >= 1.2], [4, 3, 2, 1], 0)
            + np.select([plt < 20, plt < 50, plt < 100, plt < 150], [4, 3, 2, 1], 0)
            + np.select([bil >= 12, bil >= 6, bil >= 2, bil >= 1.2], [4, 3, 2, 1], 0))
    out = pd.DataFrame({
        "qsofa": qsofa, "sirs": sirs, "sofa_partial": sofa,
        "shock_index": d["HR"] / d["SBP"].where(d["SBP"] > 0),
        "map_lt65": (d["MAP"] < 65).astype(int),
        "lactate_gt2": (d["Lactate"] > 2).astype(int),
    }, index=d.index)
    return out


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Return ID, Hour, (label) and the feature columns, one row per input row, in the input order."""
    df = df.reset_index(drop=True)
    assert df.groupby(ID)[TIME].apply(lambda s: s.is_monotonic_increasing).all(), "rows must be ordered by Hour within patient"
    g = df.groupby(ID, sort=False)
    cols: dict[str, pd.Series] = {}

    for c in DYNAMIC + [f"{c}_tsl" for c in DYNAMIC] + [f"{c}_cnt" for c in DYNAMIC] + DEMOGRAPHICS:
        cols[c] = df[c]

    roll = g[VITALS].rolling(WINDOW, min_periods=1)
    for stat in ("mean", "min", "max", "std"):
        r = getattr(roll, stat)().reset_index(level=0, drop=True).reindex(df.index)
        for c in VITALS:
            cols[f"{c}_{stat}{WINDOW}"] = r[c]

    roll12 = g[VITALS].rolling(LONG_WINDOW, min_periods=1)
    for stat in ("mean", "min", "max"):
        r = getattr(roll12, stat)().reset_index(level=0, drop=True).reindex(df.index)
        for c in VITALS:
            cols[f"{c}_{stat}{LONG_WINDOW}"] = r[c]

    for c in DYNAMIC:
        cols[f"{c}_d{LAG}"] = df[c] - g[c].shift(LAG)
    for c in VITALS:
        cols[f"{c}_d{LONG_LAG}"] = df[c] - g[c].shift(LONG_LAG)
    for c, fn in RUNNING:
        cols[f"{c}_{fn}"] = getattr(g[c], fn)()

    sc = clinical_scores(df)
    for c in sc.columns:
        cols[c] = sc[c]
    tmp = pd.concat([df[[ID]], sc], axis=1)
    gs = tmp.groupby(ID, sort=False)
    for c in ("qsofa", "sirs", "sofa_partial"):
        cols[f"{c}_max"] = gs[c].cummax()
        cols[f"{c}_d{LAG}"] = sc[c] - gs[c].shift(LAG)

    feats = pd.DataFrame(cols).astype("float32")
    keep = [ID, TIME] + ([TARGET] if TARGET in df else [])
    return pd.concat([df[keep], feats], axis=1)


def feature_columns(frame: pd.DataFrame) -> list[str]:
    return [c for c in frame.columns if c not in (ID, TIME, TARGET)]


In [ ]:
%%writefile sepsis_src/sepsis/calibration.py
"""Probability calibration. Platt scaling on the logit is strictly monotone, so it never changes the ranking
(AUROC / AUPRC are identical before and after); isotonic regression would flatten ties and lose rank information."""
import numpy as np
from sklearn.linear_model import LogisticRegression


def _logit(p: np.ndarray) -> np.ndarray:
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


class PlattCalibrator:
    def __init__(self) -> None:
        self.a, self.b = 1.0, 0.0

    def fit(self, p: np.ndarray, y: np.ndarray) -> "PlattCalibrator":
        lr = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000).fit(_logit(p).reshape(-1, 1), y)
        self.a, self.b = float(lr.coef_[0, 0]), float(lr.intercept_[0])
        return self

    def transform(self, p: np.ndarray) -> np.ndarray:
        z = self.a * _logit(p) + self.b
        return 1 / (1 + np.exp(-z))

    def to_dict(self) -> dict:
        return {"method": "platt", "a": self.a, "b": self.b}


def brier(y: np.ndarray, p: np.ndarray) -> float:
    return float(np.mean((np.asarray(p) - np.asarray(y)) ** 2))


def ece(y: np.ndarray, p: np.ndarray, bins: int = 10) -> float:
    """Expected calibration error over equal-count probability bins."""
    y, p = np.asarray(y), np.asarray(p)
    order = np.argsort(p)
    total = 0.0
    for idx in np.array_split(order, bins):
        total += len(idx) / len(p) * abs(y[idx].mean() - p[idx].mean())
    return float(total)


In [ ]:
%%writefile sepsis_src/sepsis/evaluation.py
"""Held-out evaluation: AUROC / AUPRC, curves, calibration, operating point and detection lead time."""
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve

from .config import ID, TARGET


def _thin(a: np.ndarray, n: int = 60) -> np.ndarray:
    idx = np.unique(np.linspace(0, len(a) - 1, min(n, len(a))).round().astype(int))
    return a[idx]


def operating_point(y: np.ndarray, p: np.ndarray) -> dict:
    """Threshold that maximises F1 on the held-out hours."""
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
    i = int(np.argmax(f1))
    return {"threshold": float(thr[i]), "precision": float(prec[i]), "recall": float(rec[i]), "f1": float(f1[i])}


def specificity_at(y: np.ndarray, p: np.ndarray, thr: float) -> float:
    neg = y == 0
    return float(((p < thr) & neg).sum() / neg.sum())


def calibration(y: np.ndarray, p: np.ndarray, bins: int = 10) -> list[dict]:
    q = pd.qcut(p, bins, duplicates="drop")
    g = pd.DataFrame({"y": y, "p": p, "q": q}).groupby("q", observed=True)
    return [{"predicted": float(v.p.mean()), "observed": float(v.y.mean()), "n": int(len(v))} for _, v in g]


def lead_times(df: pd.DataFrame, p: np.ndarray, thr: float) -> tuple[list[float], int, int]:
    """Hours between the first alert and clinical onset for sepsis patients.

    PhysioNet sets SepsisLabel = 1 from 6 h before clinical onset, so onset = first positive hour + 6.
    Returns (lead times of detected patients, detected, total sepsis patients).
    """
    d = pd.DataFrame({ID: df[ID].values, "h": df["Hour"].values, "y": df[TARGET].values, "p": p})
    leads, detected, total = [], 0, 0
    for _, g in d.groupby(ID):
        pos = g[g.y == 1]
        if pos.empty:
            continue
        total += 1
        onset = pos.h.min() + 6
        alerts = g[g.p >= thr]
        if not alerts.empty:
            detected += 1
            leads.append(float(onset - alerts.h.min()))
    return leads, detected, total


def evaluate(df: pd.DataFrame, p: np.ndarray, n_train_patients: int, n_train_rows: int,
             test_patients: int | None = None, test_rows: int | None = None) -> dict:
    y = df[TARGET].values
    op = operating_point(y, p)
    fpr, tpr, _ = roc_curve(y, p)
    prec, rec, _ = precision_recall_curve(y, p)
    leads, det, tot = lead_times(df, p, op["threshold"])
    return {
        "auroc": float(roc_auc_score(y, p)),
        "auprc": float(average_precision_score(y, p)),
        "baselineAuprc": float(y.mean()),
        "sensitivity": op["recall"],
        "specificity": specificity_at(y, p, op["threshold"]),
        "precision": op["precision"],
        "operatingThreshold": op["threshold"],
        "medianLeadTimeHours": float(np.median(leads)) if leads else None,
        "patientsDetected": det,
        "sepsisPatients": tot,
        "roc": [{"fpr": float(a), "tpr": float(b)} for a, b in zip(_thin(fpr), _thin(tpr))],
        "pr": [{"recall": float(a), "precision": float(b)} for a, b in zip(_thin(rec), _thin(prec))],
        "calibration": calibration(y, p),
        "leadTimes": [float(np.clip(v, -48, 48)) for v in leads],
        "partitions": [
            {"name": "Training", "patients": n_train_patients, "rows": n_train_rows, "status": "Used to fit the model"},
            {"name": "Held-out validation", "patients": int(df[ID].nunique()), "rows": int(len(df)),
             "status": "Labelled; source of every metric shown"},
            *([{"name": "Test", "patients": test_patients, "rows": test_rows,
                "status": "Unlabelled; scored for submission only, no metrics"}] if test_patients is not None else []),
        ],
    }


In [ ]:
%%writefile sepsis_src/sepsis/submission.py
"""Build and validate the submission file against the organizers' rules:
exact columns and row order of sample_submission.csv, one probability in [0, 1] per test row, nothing dropped."""
import numpy as np
import pandas as pd

from .config import ID, TIME

COLUMNS = [ID, TIME, "SepsisLabel"]


def build(sample: pd.DataFrame, scored: pd.DataFrame, prob_col: str = "SepsisLabel") -> pd.DataFrame:
    """Attach probabilities to the sample's own (Patient_ID, Hour) rows, preserving its order."""
    if scored.duplicated([ID, TIME]).any():
        raise ValueError("scored frame has duplicate (Patient_ID, Hour) keys")
    sub = sample[[ID, TIME]].merge(scored[[ID, TIME, prob_col]].rename(columns={prob_col: "SepsisLabel"}), on=[ID, TIME], how="left")
    return sub[COLUMNS]


def validate(sub: pd.DataFrame, sample: pd.DataFrame) -> None:
    """Raise AssertionError with a specific message if any submission rule is broken."""
    assert list(sub.columns) == COLUMNS, f"columns must be exactly {COLUMNS}, got {list(sub.columns)}"
    assert len(sub) == len(sample), f"row count {len(sub)} != sample {len(sample)}"
    assert (sub[ID].to_numpy() == sample[ID].to_numpy()).all() and (sub[TIME].to_numpy() == sample[TIME].to_numpy()).all(), \
        "Patient_ID / Hour must match sample_submission.csv row for row, in the same order"
    p = sub["SepsisLabel"]
    assert pd.api.types.is_float_dtype(p), "SepsisLabel must be numeric probabilities"
    assert p.notna().all(), f"{int(p.isna().sum())} rows have no prediction (never drop test rows)"
    assert np.isfinite(p.to_numpy()).all(), "SepsisLabel contains inf"
    assert p.between(0, 1).all(), "probabilities must be within [0, 1]"
    assert p.nunique() > 2, "predictions look binary or constant; submit continuous probabilities"


In [ ]:
%%writefile sepsis_src/sepsis/modeling.py
"""Snapshot sepsis model: hourly vitals, labs and demographics -> probability of the sepsis label at that hour.

Inputs are the last known (forward-filled) values, so a clinician can enter them by hand. Gradient-boosted trees
handle missing inputs natively, which is why unmeasured labs are left as NaN instead of being imputed.
"""
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit

from .config import ID, INTERIM, LABS, SEED, TARGET, VITALS

DEMOGRAPHICS = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime", "ICULOS"]
FEATURES = VITALS + LABS + DEMOGRAPHICS

# stands in for an infinite split threshold: every measured value goes left, only missing values go right
BIG = 1e30

PARAMS = dict(max_iter=250, learning_rate=0.06, max_leaf_nodes=31, min_samples_leaf=100,
              l2_regularization=1.0, early_stopping=False, random_state=SEED)


# time-aware model (rolling stats, trends, running extremes, clinical scores): see features.py and scripts/05_train_temporal.py
TEMPORAL_PARAMS = dict(max_iter=400, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=200,
                       l2_regularization=3.0, early_stopping=False, random_state=SEED)


def load(split: str) -> pd.DataFrame:
    cols = [ID, "Hour"] + FEATURES + ([TARGET] if split == "train" else [])
    return pd.read_csv(INTERIM / f"clean_{split}.csv.gz", usecols=cols)


def split_by_patient(df: pd.DataFrame, test_size: float = 0.2):
    """Patient-level holdout: no patient appears on both sides, so hours of one stay never leak."""
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED)
    tr, va = next(gss.split(df, df[TARGET], groups=df[ID]))
    return df.iloc[tr], df.iloc[va]


def fit(df: pd.DataFrame) -> HistGradientBoostingClassifier:
    return HistGradientBoostingClassifier(**PARAMS).fit(df[FEATURES], df[TARGET])


def _r(x: float) -> float:
    return float(f"{x:.8g}")


def export_trees(model: HistGradientBoostingClassifier) -> dict:
    """Plain-JSON copy of the fitted trees: the browser evaluates sum(leaf values) + baseline, then a sigmoid."""
    trees = []
    for per_class in model._predictors:
        nodes = per_class[0].nodes
        leaf = nodes["is_leaf"].astype(bool)
        trees.append({
            "f": np.where(leaf, -1, nodes["feature_idx"]).tolist(),
            "t": [_r(v) if np.isfinite(v) else BIG for v in nodes["num_threshold"]],  # inf (leaves, and splits that only separate missing values) is not valid JSON
            "l": nodes["left"].tolist(),
            "r": nodes["right"].tolist(),
            "v": [_r(v) for v in nodes["value"]],
            "m": nodes["missing_go_to_left"].astype(int).tolist(),
        })
    return {"baseline": _r(float(model._baseline_prediction.ravel()[0])), "trees": trees}


def predict_raw(tree_json: dict, X: np.ndarray) -> np.ndarray:
    """Reference implementation of the exported evaluator (used to verify the JSON against sklearn)."""
    out = np.full(len(X), tree_json["baseline"], dtype=float)
    for t in tree_json["trees"]:
        for i, row in enumerate(X):
            n = 0
            while t["f"][n] >= 0:
                x = row[t["f"][n]]
                go_left = bool(t["m"][n]) if np.isnan(x) else x <= t["t"][n]
                n = t["l"][n] if go_left else t["r"][n]
            out[i] += t["v"][n]
    return out


def feature_meta(df: pd.DataFrame) -> dict:
    """Observed range and median per feature, used by the form for hints and range warnings."""
    meta = {}
    for f in FEATURES:
        s = df[f].dropna()
        meta[f] = {"min": _r(float(s.min())), "max": _r(float(s.max())), "median": _r(float(s.median())),
                   "measured": _r(float(df[f].notna().mean()))}
    return meta


def write_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, separators=(",", ":"), allow_nan=False), encoding="utf-8")


In [ ]:
%%writefile sepsis_src/sepsis/pipeline.py
"""End-to-end time-aware pipeline, shared by scripts/05_train_temporal.py and the Kaggle notebook.

cleaned frames -> causal features -> patient-grouped CV (out-of-fold probabilities) -> calibration check ->
alert threshold -> final fit on all training patients -> calibrated test probabilities.
"""
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

from . import calibration, evaluation
from .config import ID, SEED, TARGET, TIME
from .features import add_features, feature_columns


def run(train_clean: pd.DataFrame, test_clean: pd.DataFrame, params: dict, n_folds: int, log=print) -> dict:
    train, test = add_features(train_clean), add_features(test_clean)
    cols = feature_columns(train)
    assert cols == feature_columns(test), "train and test features differ"
    y, groups = train[TARGET].to_numpy(), train[ID].to_numpy()
    log(f"features: {len(cols)} columns, train {len(train)} rows / {train[ID].nunique()} patients, test {len(test)} rows")

    # patient-grouped cross-validation -> out-of-fold probabilities
    oof = np.zeros(len(train))
    folds = []
    for k, (tr, va) in enumerate(GroupKFold(n_splits=n_folds).split(train, y, groups)):
        assert not set(groups[tr]) & set(groups[va]), "a patient appears in both train and validation"
        m = HistGradientBoostingClassifier(**params).fit(train.iloc[tr][cols], y[tr])
        oof[va] = m.predict_proba(train.iloc[va][cols])[:, 1]
        folds.append({"fold": k, "auroc": roc_auc_score(y[va], oof[va]), "auprc": average_precision_score(y[va], oof[va])})
        log(f"fold {k}: AUROC {folds[-1]['auroc']:.4f} AUPRC {folds[-1]['auprc']:.4f}")

    # calibration: fit on one half of the out-of-fold patients, judge on the other half
    a_idx, b_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED).split(train, y, groups))
    check = calibration.PlattCalibrator().fit(oof[a_idx], y[a_idx])
    raw_b, cal_b = oof[b_idx], check.transform(oof[b_idx])
    calib = {"brierRaw": calibration.brier(y[b_idx], raw_b), "brierCalibrated": calibration.brier(y[b_idx], cal_b),
             "eceRaw": calibration.ece(y[b_idx], raw_b), "eceCalibrated": calibration.ece(y[b_idx], cal_b)}
    use_calibration = calib["brierCalibrated"] <= calib["brierRaw"]
    calibrator = calibration.PlattCalibrator().fit(oof, y) if use_calibration else calibration.PlattCalibrator()
    oof_cal = calibrator.transform(oof)
    log("calibration", {k: round(v, 5) for k, v in calib.items()}, "-> using Platt" if use_calibration else "-> keeping raw probabilities")

    # metrics on all out-of-fold predictions; alert threshold = max F1 on calibrated probabilities
    ev = evaluation.evaluate(train, oof_cal, int(train[ID].nunique()), int(len(train)))
    thr = ev["operatingThreshold"]
    table = []
    for t in (0.02, 0.05, 0.10, 0.15, 0.20, 0.30):
        flagged = oof_cal >= t
        table.append({"threshold": t, "flaggedShare": float(flagged.mean()),
                      "precision": float(y[flagged].mean()) if flagged.any() else None, "recall": float(flagged[y == 1].mean())})
    metrics = {
        "model": "temporal-hgb", "features": len(cols), "cv": f"{n_folds}-fold GroupKFold on Patient_ID", "folds": folds,
        "auroc": float(roc_auc_score(y, oof)), "auprc": float(average_precision_score(y, oof)), "baselineAuprc": float(y.mean()),
        "calibration": calib, "calibrationUsed": bool(use_calibration), "calibrator": calibrator.to_dict(),
        "alertThreshold": thr, "alertThresholdRule": "maximises F1 on calibrated out-of-fold probabilities",
        "atThreshold": {k: ev[k] for k in ("sensitivity", "specificity", "precision", "medianLeadTimeHours", "patientsDetected", "sepsisPatients")},
        "thresholdTable": table,
    }
    log(f"OOF AUROC {metrics['auroc']:.4f} AUPRC {metrics['auprc']:.4f} (baseline {metrics['baselineAuprc']:.4f}); "
        f"alert threshold {thr:.3f}: sensitivity {ev['sensitivity']:.3f}, precision {ev['precision']:.3f}")

    # final model on every training patient
    final = HistGradientBoostingClassifier(**params).fit(train[cols], y)
    test_p = calibrator.transform(final.predict_proba(test[cols])[:, 1])
    log(f"test probabilities: mean {test_p.mean():.4f}, max {test_p.max():.3f}")
    return {"metrics": metrics, "evaluation": ev, "model": final, "features": cols, "calibrator": calibrator,
            "threshold": thr, "test": test[[ID, TIME]].assign(SepsisLabel=test_p)}


In [ ]:
sys.path.insert(0, str(OUT_DIR / "sepsis_src"))
from sepsis import modeling, pipeline, submission
from sepsis.cleaning import clean
from sepsis.config import ID, N_FOLDS, TIME

raw_train, raw_test = pd.read_csv(PATHS["train.csv"]), pd.read_csv(PATHS["test.csv"])
sample = pd.read_csv(PATHS["sample_submission.csv"])
print("train", raw_train.shape, "patients", raw_train[ID].nunique(), "| test", raw_test.shape, "patients", raw_test[ID].nunique())
print("sepsis rate (rows):", round(raw_train["SepsisLabel"].mean(), 4), "| patients ever septic:", round(raw_train.groupby(ID)["SepsisLabel"].max().mean(), 4))
assert not set(raw_train[ID]) & set(raw_test[ID]), "train and test share patients"
clean_train, clean_test = clean(raw_train), clean(raw_test)
print("cleaned", clean_train.shape, clean_test.shape)

## Train, cross-validate, calibrate

In [ ]:
params = dict(modeling.TEMPORAL_PARAMS)
n_folds = N_FOLDS
if FAST:
    params["max_iter"], n_folds = 40, 2
    print("FAST smoke run: results are NOT the real model")
t0 = time.time()
result = pipeline.run(clean_train, clean_test, params, n_folds, log=lambda *a: print(f"[{time.time() - t0:5.0f}s]", *a))

## Held-out (out-of-fold) results

In [ ]:
m, ev = result["metrics"], result["evaluation"]
print(f"AUROC {m['auroc']:.4f}   AUPRC {m['auprc']:.4f}   (an average hour is {m['baselineAuprc']:.4f})")
print(f"Calibration (held-out patients): Brier {m['calibration']['brierRaw']:.5f} raw vs {m['calibration']['brierCalibrated']:.5f} Platt; "
      f"ECE {m['calibration']['eceRaw']:.4f} vs {m['calibration']['eceCalibrated']:.4f}; calibration applied: {m['calibrationUsed']}")
a = m["atThreshold"]
print(f"Alert threshold {m['alertThreshold']:.3f} ({m['alertThresholdRule']}): sensitivity {a['sensitivity']:.3f}, "
      f"specificity {a['specificity']:.3f}, precision {a['precision']:.3f}; "
      f"{a['patientsDetected']} of {a['sepsisPatients']} sepsis patients alerted, median first alert {a['medianLeadTimeHours']:.1f} h before onset")
pd.DataFrame(m["thresholdTable"])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].plot([p["recall"] for p in ev["pr"]], [p["precision"] for p in ev["pr"]], color="#286b75")
ax[0].axhline(m["baselineAuprc"], ls="--", color="grey"); ax[0].set(title="Precision-recall", xlabel="Recall", ylabel="Precision")
ax[1].plot([p["fpr"] for p in ev["roc"]], [p["tpr"] for p in ev["roc"]], color="#286b75")
ax[1].plot([0, 1], [0, 1], ls="--", color="grey"); ax[1].set(title="ROC", xlabel="False positive rate", ylabel="True positive rate")
c = ev["calibration"]
ax[2].plot([b["predicted"] for b in c], [b["observed"] for b in c], "o-", color="#286b75")
top = max(b["predicted"] for b in c); ax[2].plot([0, top], [0, top], ls="--", color="grey")
ax[2].set(title="Calibration (out-of-fold)", xlabel="Predicted probability", ylabel="Observed sepsis rate")
plt.tight_layout(); plt.show()

## Submission
One probability per test row, in the order of `sample_submission.csv`.

In [ ]:
sub = submission.build(sample, result["test"])
submission.validate(sub, sample)
sub.to_csv(OUT_DIR / "submission.csv", index=False)
print(f"wrote {OUT_DIR / 'submission.csv'}: {len(sub)} rows, probabilities {sub['SepsisLabel'].min():.4f} to {sub['SepsisLabel'].max():.4f}")
sub.head()

## Notes and limits
* The features and the model only look backwards in time; test patients are scored independently of each other.
* Metrics are out-of-fold on the training patients. The hidden test labels are never used.
* Median lead time counts the first alert at the chosen threshold relative to the PhysioNet convention that `SepsisLabel` turns on 6 h before clinical onset; early alerts may include false alarms.
* This is a research model trained on public PhysioNet/CinC 2019 data and is not a medical device.